# 01 — Il dataset con le etichette NewsGuard 2026 (tentativo 11)

Gli **articoli puliti** dei tentativi 06–09 (`derived/01_articoli_puliti.csv`: pulizia della Versione 2, articoli dal
2025) con **due etichette** per testata: il livello NewsGuard **2026** (l'obiettivo) e quello **2022** (il controllo).
Stesse testate e stessi fold per le due: il confronto dice solo l'effetto dell'aggiornamento delle etichette.
Fissato il 23 ottobre, prima dei risultati. Le regole sono quelle del tentativo 09 (`1-data-cleaning/03_dataset_finale`):

1. **testate**: quelle con un punteggio sia nel 2026 sia nel 2022;
2. **gruppi** = le nostre reti (`02_reti`) ∪ stesso owner ∪ stesso Parent Domain, presi **da tutti e due i feed**
   (2022 e 2026): una proprietà cambiata in questi anni lega comunque le testate;
3. **copie tra gruppi diversi**: di un testo pubblicato da testate di gruppi diversi si tiene la copia più vecchia;
4. **valutate** (il test): almeno 20 articoli e al più 5 testate per gruppo, a caso con seed fisso, senza guardare le
   etichette; **di solo training**: le altre con almeno 5 articoli, peso min(1, 5/n);
5. **fold**: 5, per gruppo, stratificati per livello **2026**, fatti sulle valutate; le testate di solo training
   seguono il loro gruppo.

Uscite: `Dataset/versioni/dataset.csv` (articoli, con `label` = livello 2026 e `label_2022`) e `fold.csv` (una riga
per testata, con punteggi e criteri dei due anni); in `Results/2-dataset` passi e composizione.

In [ ]:
import networkx as nx
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

from comune import (COPIE, CRITERI_2022, CRITERI_2026, LIVELLI, NG2022, NG2026, PULITI, RESULTS_DATASET, RETI, SEED, VERSIONI,
                    leggi_newsguard, livello, normalizza_owner)

MIN_ARTICOLI = 20
MAX_TESTATE_PER_GRUPPO = 5
MIN_ARTICOLI_TRAINING = 5
RESULTS_DATASET.mkdir(parents=True, exist_ok=True); VERSIONI.mkdir(parents=True, exist_ok=True)
n26, n22 = leggi_newsguard(NG2026), leggi_newsguard(NG2022)
art = pd.read_csv(PULITI, parse_dates=["data"])
passi = []


def passo(nome):
    passi.append({"passo": nome, "articoli": len(art), "testate": art.domain.nunique()})
    print(f"{nome}: {len(art)} articoli, {art.domain.nunique()} testate")


passo("articoli puliti (tentativi 06-09)")
art = art[art.domain.map(n26.Score).notna() & art.domain.map(n22.Score).notna()]
passo("1. testate con un punteggio NewsGuard 2026 e 2022")

## 2. I gruppi: reti ∪ owner ∪ Parent Domain (2022 e 2026)

In [ ]:
reti = pd.read_csv(RETI).set_index("domain")
nodi = sorted(set(reti.index) | set(art.domain))
G = nx.Graph(); G.add_nodes_from(nodi)
for _, g in reti.groupby("rete"):
    G.add_edges_from(zip(g.index, g.index[1:]))
for ng in (n22, n26):
    for campo, norma in [("Owner", normalizza_owner), ("Parent Domain", lambda x: "" if str(x) == "nan" else str(x).strip().lower())]:
        s = pd.Series({d: norma(ng[campo].get(d, "")) for d in nodi})
        s = s[s != ""]
        for _, g in s.groupby(s):
            G.add_edges_from(zip(g.index, g.index[1:]))
gruppo = {d: i for i, c in enumerate(sorted(nx.connected_components(G), key=len, reverse=True)) for d in c}
art["gruppo"] = art.domain.map(gruppo); art["rete"] = art.domain.map(reti.rete)
print(f"{len(set(gruppo.values()))} gruppi; le testate del dataset stanno in {art.gruppo.nunique()} gruppi")

## 3. Copie tra gruppi diversi

In [ ]:
copie = pd.read_csv(COPIE)
presenti = set(art.article_id)
copie = copie[copie.articolo_a.isin(presenti) & copie.articolo_b.isin(presenti)]
C = nx.Graph(); C.add_edges_from(zip(copie.articolo_a, copie.articolo_b))
info = art.set_index("article_id")
via = set()
for fam in nx.connected_components(C):
    fam = sorted(fam, key=lambda x: (info.data[x], x))           # la più vecchia per prima
    if len({info.gruppo[x] for x in fam}) > 1:
        primo = info.gruppo[fam[0]]
        via |= {x for x in fam if info.gruppo[x] != primo}
art = art[~art.article_id.isin(via)]
passo("3. senza copie tra gruppi diversi")

## 4. Testate valutate e di solo training

In [ ]:
n = art.groupby("domain").size()
art = art[art.domain.map(n) >= MIN_ARTICOLI_TRAINING]
passo(f"4. testate con almeno {MIN_ARTICOLI_TRAINING} articoli (training)")
t = art.groupby("domain").agg(gruppo=("gruppo", "first"), rete=("rete", "first"), articoli=("article_id", "size"))
idonee = t[t.articoli >= MIN_ARTICOLI].reset_index()
valutate = set(idonee.sample(frac=1, random_state=SEED).groupby("gruppo").head(MAX_TESTATE_PER_GRUPPO).domain)
t["valutazione"] = t.index.isin(valutate)
t["testate_nel_gruppo"] = t.groupby("gruppo").articoli.transform("size")
t["testate_valutate_nel_gruppo"] = t.groupby("gruppo").valutazione.transform("sum")
t["peso"] = np.minimum(1.0, MAX_TESTATE_PER_GRUPPO / t.testate_nel_gruppo)
t["score"], t["score_2022"] = t.index.map(n26.Score), t.index.map(n22.Score)
t["label"], t["label_2022"] = livello(t.score.to_numpy()).to_numpy(), livello(t.score_2022.to_numpy()).to_numpy()
t["aggiornato"] = pd.to_datetime(t.index.map(n26["Last Updated"]), unit="ms").date
t["aggiornato_2022"] = pd.to_datetime(t.index.map(n22["Last Updated"]), unit="ms").date
for nome, ng, crit in [("", n26, CRITERI_2026), ("_2022", n22, CRITERI_2022)]:
    for k, c in enumerate(crit, 1):
        t[f"criterio{k}{nome}"] = t.index.map(ng[c]).map({"Yes": 1.0, "No": 0.0}).fillna(1.0)    # non applicato = prende i punti
passi.append({"passo": f"5. valutate: almeno {MIN_ARTICOLI} articoli, al più {MAX_TESTATE_PER_GRUPPO} per gruppo",
              "articoli": int(art.domain.isin(valutate).sum()), "testate": len(valutate)})
print(f"valutate: {len(valutate)} in {t[t.valutazione].gruppo.nunique()} gruppi; di solo training: {(~t.valutazione).sum()}")

## 5. I fold (per gruppo, stratificati sul livello 2026)

In [ ]:
t = t.sort_index()
val = t[t.valutazione].reset_index()
val["fold_gruppo"] = 0
for i, (_, idx) in enumerate(StratifiedGroupKFold(5, shuffle=True, random_state=SEED).split(val, val.label, val.gruppo), 1):
    val.loc[idx, "fold_gruppo"] = i
fold_del_gruppo = val.groupby("gruppo").fold_gruppo.first()
rng = np.random.default_rng(SEED)
senza = sorted(set(t.gruppo) - set(fold_del_gruppo.index))
fold_del_gruppo = pd.concat([fold_del_gruppo, pd.Series(rng.integers(1, 6, len(senza)), index=senza)])
t["fold_gruppo"] = t.gruppo.map(fold_del_gruppo).astype(int)
assert t.groupby("gruppo").fold_gruppo.nunique().max() == 1
pd.crosstab(val.fold_gruppo, val.label.map(dict(enumerate(LIVELLI))), margins=True, margins_name="tot")

## Composizione e salvataggio

In [ ]:
for c in ["label", "label_2022", "score", "score_2022", "valutazione", "peso", "fold_gruppo"]:
    art[c] = art.domain.map(t[c])
art[["article_id", "domain", "gruppo", "rete", "label", "label_2022", "score", "score_2022", "valutazione", "peso", "fold_gruppo",
     "data", "titolo", "autore", "url", "text"]].to_csv(VERSIONI / "dataset.csv", index=False)
t.reset_index(names="domain").to_csv(VERSIONI / "fold.csv", index=False)
pd.DataFrame(passi).set_index("passo").to_csv(RESULTS_DATASET / "01_passi.csv")
righe = []
for nome, sel in [("valutate", t.valutazione), ("tutte (training)", t.valutazione | True)]:
    for etichetta in ["label", "label_2022"]:
        r = {"testate": nome, "etichetta": "2026" if etichetta == "label" else "2022", "n": int(sel.sum()), "gruppi": int(t.gruppo[sel].nunique()),
             "affidabili": float((t[etichetta][sel] >= 2).mean())}
        r.update(t[etichetta][sel].map(dict(enumerate(LIVELLI))).value_counts().reindex(LIVELLI).fillna(0).astype(int).to_dict())
        righe.append(r)
comp = pd.DataFrame(righe).set_index(["testate", "etichetta"]); comp.to_csv(RESULTS_DATASET / "01_composizione.csv")
v = t[t.valutazione]
print(f"valutate con livello diverso tra 2022 e 2026: {(v.label != v.label_2022).sum()} su {len(v)}; binario diverso: {((v.label >= 2) != (v.label_2022 >= 2)).sum()}")
display(pd.DataFrame(passi).set_index("passo")); comp.round(3)